In [ ]:
import os
import socket

import torch
import torch.distributed as dist


def train_openmpi_cuda_smoke():
    import os
    import socket

    import torch
    import torch.distributed as dist

    if not dist.is_mpi_available():
        raise RuntimeError("PyTorch MPI backend is not available in this image")

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for the OpenMPI SDK smoke test")

    local_rank = int(os.environ.get("OMPI_COMM_WORLD_LOCAL_RANK", "0"))
    expected_world_size = int(os.environ.get("EXPECTED_WORLD_SIZE", "2"))

    dist.init_process_group(backend="mpi")
    torch.cuda.set_device(local_rank)

    rank = dist.get_rank()
    world_size = dist.get_world_size()
    if world_size != expected_world_size:
        raise RuntimeError(
            f"expected MPI world size {expected_world_size}, got {world_size}"
        )

    device = torch.device(f"cuda:{local_rank}")
    tensor = torch.tensor([rank + 1], device=device, dtype=torch.float32)
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)

    expected = world_size * (world_size + 1) / 2
    if float(tensor.item()) != float(expected):
        raise RuntimeError(
            f"unexpected allreduce result: got {tensor.item()}, want {expected}"
        )

    print(
        f"[Rank {rank}/{world_size}] host={socket.gethostname()} local_rank={local_rank} allreduce={tensor.item()}",
        flush=True,
    )

    dist.barrier()

    if rank == 0:
        print("MPI SDK test PASSED", flush=True)

    dist.destroy_process_group()


In [ ]:
import os

from kubernetes import client as k8s_client
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient

openshift_api_url = os.getenv("OPENSHIFT_API_URL", "")
token = os.getenv("NOTEBOOK_USER_TOKEN", "")
if not openshift_api_url or not token:
    raise RuntimeError("OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN are required")

cfg = k8s_client.Configuration()
cfg.host = openshift_api_url
cfg.verify_ssl = False
cfg.api_key = {"authorization": f"Bearer {token}"}

api_client = k8s_client.ApiClient(cfg)
backend_cfg = KubernetesBackendConfig(
    client_configuration=api_client.configuration,
)
client = TrainerClient(backend_cfg)


In [ ]:
training_runtime_name = os.getenv("TRAINING_RUNTIME", "")
if not training_runtime_name:
    raise RuntimeError("TRAINING_RUNTIME environment variable is required")

try:
    mpi_runtime = client.get_runtime(training_runtime_name)
except Exception as e:
    raise RuntimeError(f"Runtime '{training_runtime_name}' not found or not accessible") from e

kueue_queue_name = os.getenv("KUEUE_QUEUE_NAME", "")
if kueue_queue_name:
    print(f"[notebook] Kueue enabled with LocalQueue: {kueue_queue_name}")
else:
    print("[notebook] KUEUE_QUEUE_NAME not set; TrainJob will not request an explicit queue")


In [ ]:
from kubeflow.trainer import CustomTrainer
from kubeflow.trainer.options import Labels

train_options = []
if kueue_queue_name:
    train_options.append(Labels(labels={"kueue.x-k8s.io/queue-name": kueue_queue_name}))

job_name = client.train(
    trainer=CustomTrainer(
        func=train_openmpi_cuda_smoke,
        num_nodes=2,
        resources_per_node={
            "cpu": 2,
            "memory": "8Gi",
            "nvidia.com/gpu": 1,
        },
    ),
    runtime=mpi_runtime,
    options=train_options,
)

print(f"[notebook] Job submitted: {job_name}")


In [ ]:
client.wait_for_job_status(name=job_name, status={"Running", "Complete", "Failed"}, timeout=300)
client.wait_for_job_status(name=job_name, status={"Complete", "Failed"}, timeout=600)

job = client.get_job(name=job_name)
logs = list(client.get_job_logs(name=job_name, follow=False))
log_text = "\n".join(str(line) for line in logs)

print(f"Training job final status: {job.status}")

if job.status == "Failed":
    print("ERROR: MPI SDK training job failed")
    for line in logs[-50:]:
        print(line)
    raise RuntimeError(f"MPI SDK training job '{job_name}' failed")

if "MPI SDK test PASSED" not in log_text:
    print("ERROR: MPI success marker not found in logs")
    for line in logs[-80:]:
        print(line)
    raise RuntimeError("MPI SDK training did not complete successfully")

print(f"✓ MPI SDK training job '{job_name}' completed successfully")
